<img src='sharif_logo.png' alt="SUT logo" width=150 height=150 align=left class="saturate" >

<br>
<font face="Times New Roman">
<div dir=ltr align=center>
<font color=0F5298 size=7>
 Deep Learning <br>
<font color=2565AE size=5>
Computer Engineering Department - Spring 2025  <br>
<font color=3C99D size=5>
          Homework 2:  <br>
<font color=696880 size=4>
           
    

# Assignment Overview

In this assignment, you will explore inference scaling techniques in large language models (LLMs) and evaluate their performance using the Math Benchmark. Throughout the notebook, you will learn about several inference methods, including:

- **Chain-of-Thought (CoT):** A method where the model generates intermediate reasoning steps before providing the final answer.
- **Best-of-n Sampling:** An approach that generates multiple candidate responses and selects the best one based on a scoring function.
- **Beam Search:** A technique that expands several possible sequences simultaneously, choosing the most promising ones based on probability.
- **Self-Refinement:** An iterative process where the model revises its output to improve accuracy and coherence.

The **Math Benchmark** is a suite of challenging mathematical problems designed to test the reasoning and problem-solving capabilities of LLMs. The benchmark includes a variety of questions ranging from basic arithmetic and algebra to more advanced topics such as geometry and calculus. For example, you might be asked to solve an equation like `2x + 5 = 15` or compute the derivative of a function, tasks that assess the model's ability to handle both straightforward and complex mathematical queries.

By the end of this assignment, you will have:
- Gained a deeper understanding of inference time scaling methods in LLMs.
- Compared the effectiveness of different inference techniques using a rigorous math evaluation framework.

Let's dive into the notebook and begin exploring how these methods perform on a challenging set of math problems!


## vLLM: Accelerated Inference Engine for LLMs

vLLM is an open-source project designed to optimize the loading and inference of large language models. By leveraging advanced memory management techniques and dynamic batching, vLLM significantly speeds up the inference process, making it easier to deploy and experiment with LLMs even on hardware with limited resources
So we use vLLM to get results faster.

# installing Dependencies

In [43]:
!pip install vllm

In [2]:
!pip install transformers accelerate datasets

In [3]:
# !pip install --upgrade numpy
# import os
# os.kill(os.getpid(), 9)


This command launches a vLLM inference server with:
- Model: `DeepSeek-R1-Distill-Qwen-1.5B`
- Port: `8000` (default API endpoint)
- Precision: `half` (FP16) for memory efficiency
- Max context length: `3192` tokens

**Note:**  
🔹 Ensure you're using a GPU runtime (T4 or better) in Colab  
🔹 Only run the next cell if this one executes successfully


In [51]:
!vllm serve "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"   --port 8000   --dtype=half   --max-model-len 3192

2025-08-06 07:42:01.368338: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754466121.454122   30248 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754466121.475454   30248 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-08-06 07:42:01.560375: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO 08-06 07:42:14 [__init__.py:235] Automatically detected platform cuda.
INFO 08-06 07:42:22 [api_server.py:1755] 

* this cell lunches model in background using vllm

In [53]:
!nohup vllm serve "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B" --port 8000 --dtype=half --max-model-len 5192 &

nohup: appending output to 'nohup.out'


## LLM Query Function

* This Python function sends prompts to a locally-hosted LLM API and returns the generated response
* you can change max_tokens and temperature as you want



In [54]:
import requests
def get_llm_response(prompt):
    url = "http://localhost:8000/v1/chat/completions"

    payload = {
        "model": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
        "messages": [
            {
                "role": "user",
                "content": prompt
            }

        ],
    "max_tokens": 500,
    "temperature": 0.6
    }
    response = requests.post(url, json=payload)
    return response.json()['choices'][0]['message']['content'].strip()

# Test response generation
- testing model with some Math benchmark quesions

In [56]:
# TODO: Generate a response with these Math benchmark quesions
question1 = "How many positive whole-number divisors does 196 have?"
# real answer : 9
response1 = get_llm_response(question1)

question2 = "What is the distance, in units, between the points $(2, -6)$ and $(-4, 3)$? Express your answer in simplest radical form."
# real answer = 3\\sqrt{13}
response2 = get_llm_response(question2)

question3 = "Define\n\\[p = \\sum_{k = 1}^\\infty \\frac{1}{k^2} \\quad \\text{and} \\quad q = \\sum_{k = 1}^\\infty \\frac{1}{k^3}.\\]Find a way to write\n\\[\\sum_{j = 1}^\\infty \\sum_{k = 1}^\\infty \\frac{1}{(j + k)^3}\\]in terms of $p$ and $q.$"
# real answer = p - q
response3 = get_llm_response(question3)

print("Question 1:")
print(response1)
print("\nQuestion 2:")
print(response2)
print("\nQuestion 3:")
print(response3)

Question 1:
Alright, so I have this problem: "How many positive whole-number divisors does 196 have?" Hmm, okay. I remember that divisors are numbers that can divide into another number without leaving a remainder. So, for example, 1, 2, 3, up to 196 would all be divisors of 196, but I think there are more than that because 196 is a larger number. But I need to figure out exactly how many there are.

First, I think I should find the prime factorization of 196. Prime factorization is breaking down a number into the product of its prime number components. Let me recall how to do that. I think I start by dividing the number by the smallest prime number, which is 2, and keep dividing until I can't anymore, then move to the next prime number.

So, 196 divided by 2 is 98. Okay, that worked. Then 98 divided by 2 is 49. Hmm, that's still even, so I can divide by 2 again. Wait, no, 49 is odd, so 49 divided by 2 isn't a whole number. So I can only divide 196 by 2 twice. So that gives me 2². Now,

# Math Benchmark Evaluation

This cell is dedicated to evaluating the performance of inference scaling methods on the Math Benchmark dataset. The process works as follows:

- **Dataset Loading:** It loads the MATH-500 dataset, which contains a set of challenging math problems along with their correct solutions.
- **Answer Extraction:** The `extract_answer` function is used to parse and extract the final answer from the generated responses. This function specifically looks for a LaTeX-style format (using `\boxed{...}`) to reliably pinpoint the answer.
- **Normalization and Comparison:** Before comparing, both the predicted answer and the ground truth are normalized using several functions. These functions handle different mathematical expressions, such as fractions, matrices, and algebraic expressions, ensuring that the comparison is fair and accurate regardless of formatting differences.
- **Evaluation Loop:** For each problem:
  - The ground truth answer is extracted from the provided solution.
  - A response is generated by the LLM using a designated function.
  - The predicted answer is then extracted and compared against the ground truth.
  - The results for each problem, including whether the predicted answer is correct, are saved for later analysis.
- **Results Analysis:** After processing all problems, the cell aggregates the results and prints a summary, including the total number of problems evaluated, the number of correct answers, and the overall accuracy.

This evaluation method ensures that the output of each inference technique (such as Chain-of-Thought, Best-of-n, Beam Search, and Self-Refinement) is consistently measured against the Math Benchmark, without altering the original answers or evaluation logic.

**Note:**  

🔹 you don't need to modify this cell. Only rewrite the evaluation function portion then

🔹 you need to run this cell before evaluating.


In [13]:
import json
import os
import re
from typing import Dict, Optional, Union
from datasets import load_dataset
from tqdm import tqdm
import torch

# Load the MATH-500 dataset
def load_math500_dataset():
    dataset = load_dataset("HuggingFaceH4/MATH-500")["test"]
    return dataset

# Extract the last boxed answer from text
def extract_answer(response: str) -> Optional[str]:
    if not response:
        return None
    start_idx = response.rfind('\\boxed{')
    if start_idx == -1:
        return None
    brace_count = 1
    pos = start_idx + 7  # length of '\boxed{'
    while pos < len(response) and brace_count > 0:
        if response[pos] == '{':
            brace_count += 1
        elif response[pos] == '}':
            brace_count -= 1
        pos += 1
    if brace_count == 0:
        answer = response[start_idx + 7:pos - 1]
        return answer.strip()
    return None

# Normalization and comparison functions (unchanged from original)
def normalize_number(num_str: str) -> str:
    try:
        cleaned = re.sub(r'[,\$\\]|\s*(?:cm|m|kg|ft|in|lb|oz|ml|L)$|\s*\\text{[^}]+}', '', num_str).strip()
        if cleaned.startswith('.'):
            cleaned = '0' + cleaned
        num = float(cleaned)
        if abs(num) < 1 and '.' in cleaned:
            decimal_places = len(cleaned.split('.')[1])
            format_str = f"{{:.{decimal_places}f}}"
            result = format_str.format(num)
        else:
            result = str(num)
        return result
    except:
        return num_str

def numerically_equal(str1: str, str2: str) -> bool:
    try:
        return abs(float(str1) - float(str2)) < 1e-10
    except:
        return False

def normalize_fraction(fraction_str: str) -> str:
    try:
        fraction_str = fraction_str.replace('\\dfrac', '\\frac')
        fraction_str = ''.join(fraction_str.split())
        fraction_str = re.sub(r'\s*\\text{[^}]+}', '', fraction_str)
        mixed_brace = re.match(r'^\\frac(\d+)\{(\d+)\}$', fraction_str)
        if mixed_brace:
            num, den = mixed_brace.groups()
            return f"\\frac{{{num}}}{{{den}}}"
        no_braces = re.match(r'^\\frac(\d+)(\d+)$', fraction_str)
        if no_braces:
            num, den = no_braces.groups()
            return f"\\frac{{{num}}}{{{den}}}"
        if '/' in fraction_str and not any(c in fraction_str for c in '\\{}'):
            num, den = fraction_str.split('/')
            return f"\\frac{{{num.strip()}}}{{{den.strip()}}}"
        standard = re.match(r'^\\frac\{([^{}]+)\}\{([^{}]+)\}$', fraction_str)
        if standard:
            num, den = standard.groups()
            return f"\\frac{{{num}}}{{{den}}}"
    except:
        return fraction_str

def normalize_matrix_entry(entry: str) -> str:
    entry = ''.join(entry.split())
    if '/' in entry and not any(c in entry for c in '\\{}'):
        if entry.startswith('-'):
            num, den = entry[1:].split('/')
            return f"-{num.strip()}/{den.strip()}"
        else:
            num, den = entry.split('/')
            return f"{num.strip()}/{den.strip()}"
    entry = entry.replace('\\dfrac', '\\frac')
    frac_match = re.match(r'^(-)?\\frac\{(\d+)\}\{(\d+)\}$', entry)
    if frac_match:
        sign, num, den = frac_match.groups()
        sign = sign if sign else ''
        return f"{sign}{num}/{den}"
    return entry

def normalize_matrix(matrix_str: str) -> str:
    try:
        matrix_str = ''.join(matrix_str.split())
        match = re.match(r'^\\begin\{pmatrix\}(.*?)\\end\{pmatrix\}$', matrix_str)
        if not match:
            return matrix_str
        content = match.group(1)
        rows = content.split('\\\\')
        normalized_rows = []
        for row in rows:
            if '&' in row:
                entries = [normalize_matrix_entry(entry) for entry in row.split('&')]
            else:
                entries = [normalize_matrix_entry(row)]
            normalized_rows.append('&'.join(entries))
        result = "\\begin{pmatrix}" + "\\\\".join(normalized_rows) + "\\end{pmatrix}"
        return result
    except:
        return matrix_str

def normalize_algebraic_expression(expr: str) -> str:
    try:
        expr = ''.join(expr.split())
        monomial_match = re.match(r'^(-?\d*\.?\d*)?([a-zA-Z])(?:\^(-?\d+))?$', expr)
        if monomial_match:
            coeff, var, exp = monomial_match.groups()
            coeff = coeff if coeff and coeff not in ['+', '-'] else ('1' if not coeff else '-1')
            exp = exp if exp else '1'
            if coeff == '1' and exp == '1':
                return var
            elif coeff == '1':
                return f"{var}^{exp}"
            elif coeff == '-1' and exp == '1':
                return f"-{var}"
            elif coeff == '-1':
                return f"-{var}^{exp}"
            elif exp == '1':
                return f"{coeff}{var}"
            else:
                return f"{coeff}{var}^{exp}"
        pi_term_match = re.match(r'^(-?\d*\.?\d*)\\?pi$', expr)
        if pi_term_match:
            coeff = pi_term_match.group(1)
            if not coeff or coeff == '-':
                coeff = '-1' if coeff == '-' else '1'
            return f"{coeff}\\pi"
        frac_pi_match = re.match(r'^\\frac{([^{}]+)}{([^{}]+)}\\?pi$', expr)
        if frac_pi_match:
            num, den = frac_pi_match.groups()
            return f"\\frac{{{num}}}{{{den}}}\\pi"
        frac_match = re.match(r'^\\frac{([^{}]+)}{([^{}]+)}$', expr)
        if frac_match:
            num, den = frac_match.groups()
            return f"\\frac{{{num}}}{{{den}}}"
    except:
        return expr.lower()

def normalize_interval_bound(bound: str) -> str:
    if '\\infty' in bound:
        sign = '-' if bound.startswith('-') else ''
        return f"{sign}\\infty"
    return normalize_answer(bound) or bound

def normalize_interval(interval_str: str) -> str:
    try:
        interval_str = ''.join(interval_str.split())
        match = re.match(r'^\\left?([\[\(])(.*?),(.*?)\\right?([\]\)])$', interval_str)
        if not match:
            match = re.match(r'^([\[\(])(.*?),(.*?)([\]\)])$', interval_str)
            if not match:
                return interval_str
        left_bracket, left_bound, right_bound, right_bracket = match.groups()
        norm_left = normalize_interval_bound(left_bound)
        norm_right = normalize_interval_bound(right_bound)
        return f"\\left{left_bracket}{norm_left},{norm_right}\\right{right_bracket}"
    except:
        return interval_str

def normalize_ordered_tuple(tuple_str: str) -> str:
    try:
        tuple_str = tuple_str.replace('\\dfrac', '\\frac')
        tuple_str = tuple_str.replace('\\left', '').replace('\\right', '')
        tuple_str = re.sub(r'\\?\s+', '', tuple_str)
        inner = tuple_str.strip('()')
        parts = inner.split(',')
        normalized_parts = [normalize_answer(part.strip()) for part in parts if normalize_answer(part.strip())]
        return f"({','.join(normalized_parts)})"
    except:
        return None

def normalize_answer(answer: str) -> str:
    if answer is None:
        return ""
    answer = re.sub(r'\\text{[^}]+(?:inches|feet|meters|cm|m|kg|ft|in|lb|oz|ml|L|per|second|minute|hour)[^}]*}', '', answer)
    answer = re.sub(r'(?<!\\)\s+', '', answer)
    ordered_pair_match = re.match(r'^(?:\\left)?\((.*?)(?:\\right)?\)$', answer)
    if ordered_pair_match:
        content = ordered_pair_match.group(1)
        parts = content.split(',')
        normalized_parts = [normalize_answer(part) for part in parts if normalize_answer(part)]
        return f"({','.join(normalized_parts)})"
    answer = ''.join(answer.split())
    if not answer:
        return None
    pm_match = re.match(r'^(.*?)(?:\\pm|-)(.*?)$', answer)
    if pm_match:
        left, right = pm_match.groups()
        norm_left = normalize_answer(left) if left else ""
        norm_right = normalize_answer(right) if right else ""
        if norm_left or norm_right:
            return f"{norm_left}\\pm{norm_right}"
    trig_match = re.match(r'^\\(?:sin|cos|tan|cot|sec|csc)\s*([a-zA-Z])$', answer)
    if trig_match:
        variable = trig_match.group(1)
        func_name = re.match(r'^\\(.*?)(?:\s|$)', answer).group(1)
        return f"\\{func_name}{variable}"
    text_match = re.match(r'^(?:\\text{)?([A-Za-z]+)(?:})?$', answer)
    if text_match:
        return text_match.group(1).lower()
    if (answer.startswith('\\left[') or answer.startswith('\\left(') or
        answer.startswith('[') or answer.startswith('(')) and \
       (answer.endswith('\\right]') or answer.endswith('\\right)') or
        answer.endswith(']') or answer.endswith(')')):
        return normalize_interval(answer)
    if answer.startswith('\\begin{pmatrix}') and answer.endswith('\\end{pmatrix}'):
        return normalize_matrix(answer)
    answer = answer.replace('\\dfrac', '\\frac')
    if '\\frac' in answer or '/' in answer:
        return normalize_fraction(answer)
    neg_sqrt_match = re.match(r'^-\\sqrt\{?(\d+)\}?$', answer)
    if neg_sqrt_match:
        num = neg_sqrt_match.group(1)
        return f"-\\sqrt{{{num}}}"
    sqrt_match = re.match(r'^(\d*)?\\sqrt\{?(\d+)\}?$', answer)
    if sqrt_match:
        coeff, num = sqrt_match.groups()
        coeff = coeff if coeff else '1'
        return f"\\sqrt{{{num}}}" if coeff == '1' else f"{coeff}\\sqrt{{{num}}}"
    sqrt_with_coeff_match = re.match(r'^(\d+)\\sqrt\{?(\d+)\}?$', answer)
    if sqrt_with_coeff_match:
        coeff, num = sqrt_with_coeff_match.groups()
        return f"{coeff}\\sqrt{{{num}}}"
    base_match = re.match(r'^(\d+)(?:_\{?(\d+)\}?|_(\d+))$', answer)
    if base_match:
        number, base1, base2 = base_match.groups()
        base = base1 if base1 else base2
        return f"{number}_{base}"
    percent_match = re.match(r'^(\d+(?:\.\d*)?)\s*\\?%$', answer)
    if percent_match:
        return normalize_number(percent_match.group(1))
    unit_match = re.match(r'^(\d+(?:\.\d*)?)\s*(?:(?:\\[,\s])|,)?\s*(?:\\\\)?(?:\\text{(\w+)}|\\?(?:cm|m|kg|ft|in|lb|oz|ml|L))$', answer)
    if unit_match:
        return normalize_number(unit_match.group(1))
    currency_match = re.match(r'^\\?\$?([\d,]+\.?\d*)$', answer)
    if currency_match:
        return normalize_number(currency_match.group(1))
    if re.match(r'^-?[\d,]+$', answer):
        return normalize_number(answer)
    unit_match = re.match(r'^(-?[\d,]+(?:\.\d*)?)\s*(?:\\(?:mbox|text|hbox|displaystyle)\{[^}]+\})?(?:\^?\d)?$', answer)
    if unit_match:
        return normalize_number(unit_match.group(1))
    mc_match = re.match(r'^\\text{\(?([A-Za-z])\)?}$|^\(?([A-Za-z])\)?$', answer)
    if mc_match:
        return (mc_match.group(1) or mc_match.group(2)).lower()
    degree_match = re.match(r'^(-?[\d,]+(?:\.\d*)?)\s*(?:(?:\^?\\circ)|(?:{\\circ})|(?:°))?$', answer)
    if degree_match:
        return normalize_number(degree_match.group(1))
    answer = re.sub(r'\\text{([^{}]+)}', r'\1', answer)
    try:
        return normalize_algebraic_expression(answer)
    except:
        pass
    answer = answer.replace('\\left', '').replace('\\right', '')
    answer = answer.replace('\\(', '(').replace('\\)', ')')
    answer = answer.replace('\\[', '[').replace('\\]', ']')
    answer = answer.replace('\\{', '{').replace('\\}', '}')
    answer = re.sub(r'\\sqrt\{?(\d+)\}?', r'\\sqrt{\1}', answer)
    answer = re.sub(r'\\sqrt{([^{}]+)}', r'\\sqrt\1', answer)
    if re.match(r'^\d+\\%$', answer) or re.match(r'^\d+$', answer):
        answer = re.sub(r'\\%$', '', answer)
    answer = re.sub(r'\\text{([^{}]+)}', r'\1', answer)
    while len(answer) >= 2 and answer[0] == '{' and answer[-1] == '}':
        if '\\frac' in answer:
            break
        answer = answer[1:-1]
    return answer.lower() if answer else None

def compare_answers(correct_answer: str, predicted_answer: Optional[str]) -> bool:
    if predicted_answer is None:
        return False
    if numerically_equal(correct_answer, predicted_answer):
        return True
    normalized_correct = normalize_answer(correct_answer)
    normalized_predicted = normalize_answer(predicted_answer)
    if not normalized_correct or not normalized_predicted:
        return False
    if normalized_correct == "" and normalized_predicted == "":
        return False
    if ('\\left[' in normalized_correct or '\\left(' in normalized_correct) and \
       ('\\left[' in normalized_predicted or '\\left(' in normalized_predicted):
        return normalized_correct == normalized_predicted
    return normalized_correct == normalized_predicted

# Load existing results
def load_existing_results(filename: str) -> list[Dict]:
    try:
        with open(filename, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        return []

# Save a single result
def save_result(filename: str, result: Dict):
    results = load_existing_results(filename)
    results.append(result)
    with open(filename, 'w') as f:
        json.dump(results, f, indent=2)

# Analyze and print results
def analyze_results(results: list[Dict]):
    total = len(results)
    correct = sum(1 for r in results if r['is_correct'])
    accuracy = correct / total if total > 0 else 0
    print("\n=== Results Summary ===")
    print(f"Total problems: {total}")
    print(f"Correct answers: {correct}")
    print(f"Accuracy: {accuracy:.2%}")
    print("\n=== Incorrect Problems ===")
    for r in results:
        if not r['is_correct']:
            print(f"Problem {r['index']}:")
            print(f"Expected: {r['correct_answer']}")
            print(f"Predicted: {r['predicted_answer']}")
            print("---")

# Main evaluation function
def evaluate():
    os.makedirs("results", exist_ok=True)
    results_file = "evaluation_results_math500_deepseek.json"
    dataset = load_math500_dataset()
    existing_results = load_existing_results(results_file)
    processed_indexes = {result['index'] for result in existing_results}
    cnt = 0
    t=0
    for idx, item in enumerate(tqdm(dataset, desc="Evaluating problems")):
        if idx in processed_indexes:
            continue
        t += 1
        problem_text = item['problem']
        correct_answer = extract_answer(item['solution'])
        response = get_llm_response(problem_text)
        predicted_answer = extract_answer(response)
        is_correct = compare_answers(correct_answer, predicted_answer)
        result = {
            "index": idx,
            "problem": problem_text,
            "response": response,
            "correct_answer": correct_answer,
            "predicted_answer": predicted_answer,
            "is_correct": is_correct
        }
        save_result(results_file, result)
        if is_correct:
          cnt += 1
        print(f"cnt :  {cnt} idx: {t}")
    final_results = load_existing_results(results_file)
    analyze_results(final_results)



# Customizable CoT Prompt Template
* modify cot prompt then evaluate on math benchmark


In [14]:
COT_PROMPT = """
Let's solve the following math problem step-by-step like a math teacher helping a student.

At the end, write your final answer using the following format:

\\[
\\boxed{your_final_answer}
\\]

Problem:
"""

* generate response with cot prompt

In [ ]:
import requests

def get_COT_response(problem):
    prompt = COT_PROMPT + "\n" + problem
    url = "http://localhost:8000/v1/chat/completions"

    payload = {
        "model": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
        "messages": [
            {
                "role": "user",
                "content": prompt
            }

        ],
    "max_tokens": 1900,
    "temperature": 0.3
    }
    response = requests.post(url, json=payload)
    return response.json()['choices'][0]['message']['content'].strip()

# Evaluate CoT
* modify response generation part to evalute this method.

In [ ]:
def evaluate_cot():
    os.makedirs("results", exist_ok=True)
    results_file = "evaluation_results_math500_deepseek_cot.json"
    dataset = load_math500_dataset()
    existing_results = load_existing_results(results_file)
    processed_indexes = {result['index'] for result in existing_results}
    cnt = 0
    for idx, item in enumerate(tqdm(dataset, desc="Evaluating problems")):
        if idx in processed_indexes:
            continue
        if idx >= 30:
          break
        problem_text = item['problem']
        correct_answer = extract_answer(item['solution'])

        # TODO: Generate a response with cot
        response = get_COT_response(problem_text)
        predicted_answer = extract_answer(response)
        ##########################################################
        is_correct = compare_answers(correct_answer, predicted_answer)
        result = {
            "index": idx,
            "problem": problem_text,
            "response": response,
            "correct_answer": correct_answer,
            "predicted_answer": predicted_answer,
            "is_correct": is_correct
        }
        save_result(results_file, result)
        if is_correct:
          cnt += 1
        print(f"corrects :  {cnt} idx: {idx}")
    final_results = load_existing_results(results_file)
    analyze_results(final_results)

In [ ]:
evaluate_cot()

Evaluating problems:   0%|          | 1/500 [00:08<1:09:52,  8.40s/it]

corrects :  1 idx: 0


Evaluating problems:   0%|          | 2/500 [00:43<3:18:00, 23.86s/it]

corrects :  1 idx: 1


Evaluating problems:   1%|          | 3/500 [00:54<2:29:37, 18.06s/it]

...

corrects :  10 idx: 27


Evaluating problems:   6%|▌         | 29/500 [10:45<2:22:22, 18.14s/it]

corrects :  10 idx: 28


Evaluating problems:   6%|▌         | 30/500 [10:52<2:50:18, 21.74s/it]

corrects :  11 idx: 29

=== Results Summary ===
Total problems: 30
Correct answers: 11
Accuracy: 36.67%

=== Incorrect Problems ===
Problem 1:
Expected: p - q
Predicted: None
---
Problem 2:
Expected: \frac{14}{3}
Predicted: \dfrac{11}{3}
---
Problem 4:
Expected: \text{Evelyn}
Predicted: None
---
Problem 9:
Expected: 4
Predicted: None
---
Problem 11:
Expected: \frac{3}{56}
Predicted: None
---
Problem 12:
Expected: 284
Predicted: 254
---
Problem 14:
Expected: \sqrt{51}
Predicted: None
---
Problem 15:
Expected: 6 - 5i
Predicted: None
---
Problem 17:
Expected: \pi
Predicted: None
---
Problem 18:
Expected: 28
Predicted: your_final_answer
---
Problem 19:
Expected: 3
Predicted: None
---
Problem 20:
Expected: 6+9i
Predicted: 6 + 9i
---
Problem 21:
Expected: 13535
Predicted: None
---
Problem 22:
Expected: 5
Predicted: 4
---
Problem 23:
Expected: x=5
Predicted: 5
---
Problem 24:
Expected: 10
Predicted: 4.19\%
---
Problem 25:
Expected: 1,-2
Predicted: None
---
Problem 26:
Expected: 144
Predicted:

## Best-of-N

The Best-of-N approach generates several candidate responses for a problem and then selects the one with the highest average token log-likelihood. This ensures that the final answer, formatted within the `\boxed{}` command, is not only correct in presentation but also statistically the most reliable.


In [ ]:
def best_of_n_response(problem, N=5):
    best_answer = None
    best_avg_likelihood = float('-inf')
    best_responses = []
    prompt = COT_PROMPT + "\n" + problem

    all_generated_responses = []

    for t in range(N):

        url = "http://localhost:8000/v1/completions"
        payload = {
            "model": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
            "prompt": prompt,
            "max_tokens": 1024,
            "temperature": 0.3,
            "logprobs": 1,
        }
        response = requests.post(url, json=payload).json()

        if 'choices' not in response or not response['choices']:
            continue

        choice = response['choices'][0]
        text = choice.get('text')

        if 'logprobs' not in choice or choice['logprobs'] is None:
            continue

        token_logprobs_raw = choice['logprobs'].get('token_logprobs', [])

        valid_logprobs = [lp for lp in token_logprobs_raw if lp is not None]

        if not valid_logprobs:
            avg_logprob = -float('inf')
        else:
            avg_logprob = sum(valid_logprobs) / len(valid_logprobs)

        answer = extract_answer(text)

        all_generated_responses.append({
            'text': text,
            'answer': answer,
            'avg_logprob': avg_logprob
        })

    grouped_by_answer = {}
    for res in all_generated_responses:
        ans = res['answer']
        if ans is None:
            continue
        if ans not in grouped_by_answer:
            grouped_by_answer[ans] = []
        grouped_by_answer[ans].append(res)

    for answer, group_of_responses in grouped_by_answer.items():
        avg_likelihood_for_group = sum(r['avg_logprob'] for r in group_of_responses) / len(group_of_responses)

        if avg_likelihood_for_group > best_avg_likelihood:
            best_avg_likelihood = avg_likelihood_for_group
            best_answer = answer

    return best_answer


# Evaluate best of n

* modify response generation part to evalute this method.

In [ ]:
def evaluate_best_of_n():
    os.makedirs("results", exist_ok=True)
    results_file = "evaluation_results_math500_deepseek_best_of_n.json"
    dataset = load_math500_dataset()
    existing_results = load_existing_results(results_file)
    processed_indexes = {result['index'] for result in existing_results}
    cnt = 0
    for idx, item in enumerate(tqdm(dataset, desc="Evaluating problems")):
        if idx in processed_indexes:
            continue
        if idx >= 30:
          break
        problem_text = item['problem']
        correct_answer = extract_answer(item['solution'])
        # TODO: ##########################################################
        response = best_of_n_response(problem_text, N=5)
        predicted_answer = response
        ##########################################################
        is_correct = compare_answers(correct_answer, predicted_answer)
        result = {
            "index": idx,
            "problem": problem_text,
            "response": response,
            "correct_answer": correct_answer,
            "predicted_answer": predicted_answer,
            "is_correct": is_correct
        }
        save_result(results_file, result)
        if is_correct:
          cnt += 1
        print(f"corrects :  {cnt} idx: {idx}")
    final_results = load_existing_results(results_file)
    analyze_results(final_results)

In [ ]:
evaluate_best_of_n()

Evaluating problems:   0%|          | 1/500 [01:35<13:15:18, 95.63s/it]

corrects :  1 idx: 0


Evaluating problems:   0%|          | 2/500 [03:01<12:27:57, 90.11s/it]

corrects :  2 idx: 1


Evaluating problems:   1%|          | 3/500 [04:28<12:13:45, 88.58s/it]

...

corrects :  13 idx: 27


Evaluating problems:   6%|▌         | 29/500 [41:36<10:38:34, 81.35s/it]

corrects :  13 idx: 28


Evaluating problems:   6%|▌         | 30/500 [43:02<11:14:18, 86.08s/it]

corrects :  14 idx: 29

=== Results Summary ===
Total problems: 30
Correct answers: 14
Accuracy: 46.67%

=== Incorrect Problems ===
Problem 7:
Expected: 90^\circ
Predicted: None
---
Problem 9:
Expected: 4
Predicted: 2
---
Problem 10:
Expected: 2220
Predicted: None
---
Problem 11:
Expected: \frac{3}{56}
Predicted: None
---
Problem 12:
Expected: 284
Predicted: None
---
Problem 14:
Expected: \sqrt{51}
Predicted: None
---
Problem 15:
Expected: 6 - 5i
Predicted: 4 - \frac{3\sqrt{2}}{2} - 7i + \frac{3\sqrt{2}}{2}i
---
Problem 16:
Expected: -50
Predicted: None
---
Problem 17:
Expected: \pi
Predicted: None
---
Problem 18:
Expected: 28
Predicted: 28^{\circ}
---
Problem 20:
Expected: 6+9i
Predicted: 6 + 9i
---
Problem 23:
Expected: x=5
Predicted: None
---
Problem 24:
Expected: 10
Predicted: 49.5\%
---
Problem 25:
Expected: 1,-2
Predicted: None
---
Problem 26:
Expected: 144
Predicted: 708
---
Problem 28:
Expected: -2 + 7i
Predicted: -2 + 7i
---


## Beam Search

This cell implements a beam search strategy for generating candidate reasoning chains. The method generates multiple continuations at each reasoning step, scoring each candidate based on its average token log-likelihood. By retaining and expanding only the top candidates, the approach efficiently searches for the most promising chain-of-thought that leads to the final answer in the required format.


In [ ]:
def call_qwen_model_raw(prompt,step_num, temperature=0.8):
    """
    Sends a request to the local Qwen endpoint and returns the generated text
    along with the average token log-probability.
    """
    # Build the prompt. We assume the sample already contains the SYSTEM_PROMPT. you can modify max_tokens for different steps

    payload = {
        "model": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
        "max_tokens": 1024,
        "logprobs": True
    }
    url = "http://localhost:8000/v1/chat/completions"

    # TODO: Send a request to the Qwen model and get the response
    response = requests.post(url, json=payload).json()
    if 'choices' not in response: return "[ERROR] No valid answer was generated.", 0.0, 0
    output_text = response['choices'][0]['message']['content']

    # TODO:  Iterate over each choice in the response and append lobprob of each tocken to token_logprobs
    # TODO: Calculate the average log-likelihood

    logprob_data = response['choices'][0].get('logprobs')
    if logprob_data and 'token_logprobs' in logprob_data:
        token_logprobs = logprob_data['token_logprobs']
        avg_token_prob = sum(token_logprobs) / len(token_logprobs)
        num_tokens = len(token_logprobs)
    else:
        # If logprobs not available, use dummy values
        avg_token_prob = 0.0
        num_tokens = len(output_text.split())

    return output_text, avg_token_prob, num_tokens


class BeamCandidate:
    def __init__(self, sequence, cumulative_log_prob, step_scores, finished=False,num_token = 0):
        self.sequence = sequence
        self.cumulative_log_prob = cumulative_log_prob
        self.step_scores = step_scores
        self.finished = finished
        self.num_token = num_token

    def __repr__(self):
        return (f"BeamCandidate(score={self.cumulative_log_prob:.3f}, finished={self.finished}, "
                f"sequence={self.sequence})")



def generate_reasoning_steps(context, step_num, top_k):
    """
    For a given candidate reasoning chain (context), generate top_k candidate continuations
    for the current reasoning step (from 1 to 5). Each candidate is verified using the average
    token logprob as a proxy for quality.
    """

    # TODO: each step should have a different prompt and the prompt should be added to the context so make a prompt for each step that explains what the step is about
    candidates = []
    for i in range(top_k):
        if step_num == 1:
            candidate_prompt = context + "\n\nStep 1: Start solving the problem with the first logical step."
        elif step_num == 2:
            candidate_prompt = context + "\n\nStep 2: Continue the reasoning based on step 1."
        elif step_num == 3:
            candidate_prompt = context + "\n\nStep 3: Refine the solution and work towards the answer."
        elif step_num == 4:
            candidate_prompt = context + "\n\nStep 4: Almost done. Wrap up calculations and prepare for the final answer."
        else:
            candidate_prompt = context + "\n\nStep 5: Give the final answer in the correct format."


        # TODO: call the qwen model to get the output and avg_token_prob


        output_text, avg_token_prob, num_token = call_qwen_model_raw(candidate_prompt, step_num)
        candidate_step = output_text.strip()
        finished = "\\boxed{" in candidate_step
        candidates.append((candidate_step, avg_token_prob, num_token, finished))

    return candidates

def beam_search(init_problem_prompt, beam_width=3, max_steps=5, top_k=2):
    """
    Implements a beam search over reasoning steps.
    """
    prompt = init_problem_prompt
    initial_candidate = BeamCandidate(prompt, 0.0, [], finished=False, num_token=0)
    beams = [initial_candidate]


    for step_num in range(1, max_steps+1):
        new_beams = []

        for candidate in beams:
            if candidate.finished:
                # TODO: Propagate finished candidates unchanged.
                new_beams.append(candidate)
                continue

            step_candidates = generate_reasoning_steps(candidate.sequence, step_num, top_k)
            for (step_text, score,num_token, finished) in step_candidates:
                # TODO: Create a new candidate by appending the step text to the current sequence and updating the log-probability by averaging all token_logprobs after the new step.

                new_sequence = candidate.sequence + "\n" + step_text
                total_score = candidate.cumulative_log_prob + (score * num_token)
                new_candidate = BeamCandidate(
                    new_sequence,
                    total_score,
                    candidate.step_scores + [score],
                    finished=finished,
                    num_token=candidate.num_token + num_token
                )
                new_beams.append(new_candidate)

        if not new_beams:
            break
        # TODO: sort the new beams based on the cumulative_log_prob and put the top beam_width beams in the beams list
        new_beams.sort(key=lambda b: b.cumulative_log_prob, reverse=True)
        beams = new_beams[:beam_width]


        if all(beam.finished for beam in beams):
            break
    # TODO: Get the best candidate from the beams list that is finished
    finished_beams = [beam for beam in beams if beam.finished]
    if finished_beams:
        best_candidate = max(finished_beams, key=lambda b: b.cumulative_log_prob)
    else:
        best_candidate = max(beams, key=lambda b: b.cumulative_log_prob)

    return best_candidate

def run_qwen_beam_search(problem,beam_width, max_steps, top_k, log_level):
    """
    sets up the sample prompt, performs beam search,
    and extracts the final answer.
    """
    # TODO: Set the initial prompt to the problem and run the beam search to get the best candidate
    initial_prompt = problem
    best_candidate = beam_search(initial_prompt, beam_width, max_steps, top_k)
    final_answer = best_candidate.sequence

    return final_answer


# Evaluate beam search
* modify response generation part to evalute this method.

In [ ]:
def evaluate_beam_search():
    os.makedirs("results", exist_ok=True)
    results_file = "evaluation_results_math500_deepseek_beam_search.json"
    dataset = load_math500_dataset()
    existing_results = load_existing_results(results_file)
    processed_indexes = {result['index'] for result in existing_results}
    cnt = 0
    for idx, item in enumerate(tqdm(dataset, desc="Evaluating problems")):
        if idx in processed_indexes :
            continue
        if idx >= 30:
          break
        problem_text = item['problem']
        correct_answer = extract_answer(item['solution'])
        ##########################################################
        # TODO: Generate a response with beam search
        response = run_qwen_beam_search(problem_text, beam_width=3, max_steps=5, top_k=2, log_level="INFO")
        predicted_answer = extract_answer(response)
        ##########################################################
        is_correct = compare_answers(correct_answer, predicted_answer)
        result = {
            "index": idx,
            "problem": problem_text,
            "response": response,
            "correct_answer": correct_answer,
            "predicted_answer": predicted_answer,
            "is_correct": is_correct
        }
        save_result(results_file, result)
        if is_correct:
          cnt += 1
        print(f"corrects :  {cnt} idx: {idx}")
    final_results = load_existing_results(results_file)
    analyze_results(final_results)

In [ ]:
evaluate_beam_search()

Evaluating problems:   0%|          | 1/500 [00:14<1:56:34, 14.02s/it]

corrects :  0 idx: 0


Evaluating problems:   0%|          | 2/500 [05:57<28:43:12, 207.62s/it]

corrects :  0 idx: 1


Evaluating problems:   1%|          | 3/500 [06:21<17:07:30, 124.04s/it]

...

corrects :  15 idx: 27


Evaluating problems:   6%|▌         | 29/500 [1:40:30<27:19:45, 208.89s/it]

corrects :  15 idx: 28


Evaluating problems:   6%|▌         | 30/500 [1:41:40<26:33:00, 203.36s/it]

corrects :  16 idx: 29

=== Results Summary ===
Total problems: 30
Correct answers: 16
Accuracy: 53.33%

=== Incorrect Problems ===
Problem 0:
Expected: \left( 3, \frac{\pi}{2} \right)
Predicted: \left(3,\ \frac{\pi}{2}\right)
---
Problem 1:
Expected: p - q
Predicted: None
---
Problem 9:
Expected: 4
Predicted: None
---
Problem 11:
Expected: \frac{3}{56}
Predicted: None
---
Problem 12:
Expected: 284
Predicted: None
---
Problem 15:
Expected: 6 - 5i
Predicted: None
---
Problem 18:
Expected: 28
Predicted: None
---
Problem 20:
Expected: 6+9i
Predicted: 6 + 9i
---
Problem 21:
Expected: 13535
Predicted: None
---
Problem 22:
Expected: 5
Predicted: None
---
Problem 23:
Expected: x=5
Predicted: 5
---
Problem 25:
Expected: 1,-2
Predicted: 2
---
Problem 26:
Expected: 144
Predicted: None
---
Problem 28:
Expected: -2 + 7i
Predicted: -2 + 7i
---


## Self-Refinement

This approach begins by generating an initial solution using the given prompt. It then iteratively refines this output by providing the model with targeted feedback and asking it to improve its response. The process continues until the feedback indicates that no further refinement is necessary, ensuring that the final answer—properly formatted within the `\boxed{}` command—is as accurate and well-reasoned as possible.

In [57]:
SYSTEM_PROMPT = '''You are solving mathematics problems.

Please think step by step.

Important: Always end your solution with the final answer in this format:

\\[
\\boxed{your_answer_here}
\\]

The entire answer should be contained completely within the \\boxed{} command.'''


def generate_content(prompt):
    """
    Sends a prompt to the LLM and returns the content of the response.
    This version is corrected to not send the 'logprobs' parameter and includes error handling.
    """
    url = "http://localhost:8000/v1/chat/completions"

    payload = {
        "model": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.7,
        "max_tokens": 1024,
    }

    try:
        response = requests.post(url, json=payload).json()
        if 'choices' in response and response['choices']:
            return response['choices'][0]['message']['content'].strip()
        else:
            print("Error: Invalid response format from API:", response)
            return "Error: Could not generate content."
    except requests.exceptions.RequestException as e:
        print(f"Error: API request failed: {e}")
        return "Error: API request failed."

def extract_answer(output_text):
    if not output_text: return ""
    match = re.search(r'\\boxed\{(.+?)\}', output_text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return ""


def self_refine(problem, max_iter=2):
    """
    Generates a solution and iteratively refines it.
    Returns the final full response text AND the extracted answer.
    """
    prompt = SYSTEM_PROMPT + "\n" + problem
    current_output = generate_content(prompt)

    for iteration in range(max_iter):
        feedback_prompt = (
            "You are a math reviewer. Carefully read the following problem and solution.\n"
            "Determine if the solution is correct and well-reasoned.\n"
            "If it is correct, write:\n"
            "NEEDS_REFINE: no\n"
            "If it is incorrect or incomplete, write:\n"
            "NEEDS_REFINE: yes\n"
            "Then explain why.\n\n"
            f"Problem:\n{problem}\n\n"
            f"Solution:\n{current_output}"
        )
        feedback = generate_content(feedback_prompt)

        if "needs_refine: no" in feedback.lower():
            break

        refine_prompt = (
            "You are revising a math solution based on feedback.\n"
            "The goal is to correct and improve the solution while keeping it step-by-step.\n"
            "Always end with the final answer in the format:\n"
            "\\[\n\\boxed{your_answer_here}\n\\]\n\n"
            f"Problem:\n{problem}\n\n"
            f"Initial Solution:\n{current_output}\n\n"
            f"Feedback:\n{feedback}\n\n"
            "Please write an improved solution:"
        )
        current_output = generate_content(refine_prompt)

    answer = extract_answer(current_output)

    return current_output, answer


# Evaluate Self-Refinement
* modify response generation part to evalute this method.

In [58]:
def evaluate_self_refiner():
    os.makedirs("results", exist_ok=True)
    results_file = "evaluation_results_math500_deepseek_self_refiner.json"
    dataset = load_math500_dataset()
    existing_results = load_existing_results(results_file)
    processed_indexes = {result['index'] for result in existing_results}
    cnt = 0
    for idx, item in enumerate(tqdm(dataset, desc="Evaluating problems")):
        if idx in processed_indexes :
            continue
        if idx >= 30:
          break
        problem_text = item['problem']
        correct_answer = extract_answer(item['solution'])
        ##########################################################
        # TODO: Generate a response with self_refine
        response, predicted_answer = self_refine(problem_text)

        ##########################################################
        is_correct = compare_answers(correct_answer, predicted_answer)
        result = {
            "index": idx,
            "problem": problem_text,
            "response": response,
            "correct_answer": correct_answer,
            "predicted_answer": predicted_answer,
            "is_correct": is_correct
        }
        save_result(results_file, result)
        if is_correct:
          cnt += 1
        print(f"corrects :  {cnt} idx: {idx}")
    final_results = load_existing_results(results_file)
    analyze_results(final_results)

In [59]:
evaluate_self_refiner()

Evaluating problems:   0%|          | 1/500 [00:28<3:57:00, 28.50s/it]

corrects :  0 idx: 0


Evaluating problems:   0%|          | 2/500 [01:01<4:20:17, 31.36s/it]

corrects :  0 idx: 1


Evaluating problems:   1%|          | 3/500 [02:00<6:04:39, 44.02s/it]

...

corrects :  11 idx: 27


Evaluating problems:   6%|▌         | 29/500 [33:44<9:58:08, 76.20s/it]

corrects :  11 idx: 28


Evaluating problems:   6%|▌         | 30/500 [34:30<9:00:39, 69.02s/it]

corrects :  12 idx: 29

=== Results Summary ===
Total problems: 30
Correct answers: 12
Accuracy: 40.00%

=== Incorrect Problems ===
Problem 0:
Expected: \left( 3, \frac{\pi
Predicted: \left(3, \frac{\pi
---
Problem 1:
Expected: p - q
Predicted: 
---
Problem 2:
Expected: \frac{14
Predicted: \dfrac{14
---
Problem 7:
Expected: 90^\circ
Predicted: 
---
Problem 9:
Expected: 4
Predicted: 
---
Problem 11:
Expected: \frac{3
Predicted: 
---
Problem 12:
Expected: 284
Predicted: 
---
Problem 15:
Expected: 6 - 5i
Predicted: 
---
Problem 16:
Expected: -50
Predicted: 
---
Problem 18:
Expected: 28
Predicted: 
---
Problem 20:
Expected: 6+9i
Predicted: 6 + 9i
---
Problem 21:
Expected: 13535
Predicted: 
---
Problem 22:
Expected: 5
Predicted: 
---
Problem 23:
Expected: x=5
Predicted: 5
---
Problem 24:
Expected: 10
Predicted: 
---
Problem 25:
Expected: 1,-2
Predicted: 1
---
Problem 26:
Expected: 144
Predicted: 
---
Problem 28:
Expected: -2 + 7i
Predicted: 
---
